# Занятие 3_3_1_Загрузка и интеграция данных из различных форматов Пример 1

## Практическая работа для начинающих

На занятии мы не используем заранее подготовленные таблицы. Мы самостоятельно:

1. создадим три небольшие таблицы в Python;
2. сохраним их в CSV, Excel и JSON;
3. удалим таблицы из памяти;
4. загрузим данные обратно из файлов;
5. проверим структуру данных;
6. объединим таблицы по общим столбцам;
7. найдём запись, для которой не найден клиент;
8. выполним простой расчёт;
9. сохраним итоговый результат.

### Учебная ситуация

Мы работаем с обращениями в службу поддержки. Основная таблица содержит обращения, а две дополнительные таблицы содержат сведения о клиентах и каналах связи.

### Важное правило

Выполняйте ячейки последовательно сверху вниз. После каждой ячейки смотрите на результат и только затем переходите дальше.

## 1. Подключаем библиотеки

Для этой работы нужны:

- `pandas` — для создания и обработки таблиц;
- `Path` — для работы с папками и файлами;
- `sys` — чтобы посмотреть версию Python;
- `openpyxl` — для Excel. Если библиотека отсутствует, notebook автоматически использует CSV как резервный вариант.

In [ ]:
# Подключаем модуль sys, чтобы узнать версию Python.
import sys

# Подключаем Path для удобной работы с путями к папкам и файлам.
from pathlib import Path

# Подключаем pandas и используем короткое общепринятое имя pd.
import pandas as pd

# Считаем, что работа с Excel доступна, пока не обнаружена ошибка импорта.
excel_available = True

# Пытаемся подключить библиотеку openpyxl, которая нужна pandas для XLSX-файлов.
try:
    # Импортируем openpyxl.
    import openpyxl

    # Сообщаем, что библиотека доступна.
    print("openpyxl доступен. Будем использовать формат XLSX.")

# Если библиотека не установлена, выполняется этот блок.
except ImportError:
    # Запоминаем, что Excel недоступен.
    excel_available = False

    # Сообщаем о резервном варианте.
    print("openpyxl не найден. Справочник клиентов будет сохранён в CSV.")

# Показываем версию Python.
print("Версия Python:", sys.version.split()[0])

# Показываем версию pandas.
print("Версия pandas:", pd.__version__)

## 2. Создаём рабочие папки

В папке `data` будут храниться созданные исходные файлы. В папке `outputs` будет храниться итоговая объединённая таблица.

In [ ]:
# Получаем путь к папке, из которой запущен notebook.
project_dir = Path.cwd()

# Создаём путь к папке с исходными данными.
data_dir = project_dir / "data"

# Создаём путь к папке с результатами.
output_dir = project_dir / "outputs"

# Создаём папку data, если её ещё нет.
data_dir.mkdir(exist_ok=True)

# Создаём папку outputs, если её ещё нет.
output_dir.mkdir(exist_ok=True)

# Показываем текущую рабочую папку.
print("Рабочая папка:", project_dir)

# Показываем папку исходных данных.
print("Папка с данными:", data_dir)

# Показываем папку результатов.
print("Папка результатов:", output_dir)

## 3. Создаём таблицу обращений

Одна строка — одно обращение. Значение `КЛ-4` специально отсутствует в справочнике клиентов. Позже мы увидим, как это влияет на объединение.

In [ ]:
# Создаём словарь с данными об обращениях.
tickets_data = {
    # Указываем уникальные коды обращений.
    "Код обращения": ["ОБР-1", "ОБР-2", "ОБР-3", "ОБР-4", "ОБР-5"],

    # Указываем даты создания обращений как текст.
    "Дата обращения": ["2026-06-01", "2026-06-01", "2026-06-02", "2026-06-03", "2026-06-03"],

    # Указываем коды клиентов.
    "Код клиента": ["КЛ-1", "КЛ-2", "КЛ-1", "КЛ-3", "КЛ-4"],

    # Указываем коды каналов связи.
    "Код канала": ["КАН-1", "КАН-2", "КАН-1", "КАН-3", "КАН-2"],

    # Указываем категории обращений.
    "Категория": ["Оплата", "Доставка", "Доступ", "Оплата", "Доставка"],

    # Указываем текущий статус каждого обращения.
    "Статус": ["Решено", "В работе", "Решено", "Решено", "В работе"],

    # Указываем время решения в часах. Для незавершённых обращений используем ноль.
    "Время решения, часы": [2, 0, 5, 3, 0]
}

# Преобразуем словарь в таблицу pandas DataFrame.
tickets = pd.DataFrame(tickets_data)

# Показываем созданную таблицу.
display(tickets)

## 4. Создаём справочник клиентов

Одна строка — один клиент. Код `КЛ-4` намеренно отсутствует.

In [ ]:
# Создаём словарь со сведениями о клиентах.
clients_data = {
    # Указываем уникальные коды клиентов.
    "Код клиента": ["КЛ-1", "КЛ-2", "КЛ-3"],

    # Указываем сегмент каждого клиента.
    "Сегмент клиента": ["Физическое лицо", "Малый бизнес", "Физическое лицо"],

    # Указываем город клиента.
    "Город": ["Москва", "Казань", "Самара"]
}

# Преобразуем словарь в таблицу pandas DataFrame.
clients = pd.DataFrame(clients_data)

# Показываем справочник клиентов.
display(clients)

## 5. Создаём справочник каналов

Одна строка — один канал связи.

In [ ]:
# Создаём словарь со сведениями о каналах связи.
channels_data = {
    # Указываем уникальные коды каналов.
    "Код канала": ["КАН-1", "КАН-2", "КАН-3"],

    # Указываем понятные названия каналов.
    "Название канала": ["Телефон", "Электронная почта", "Чат"]
}

# Преобразуем словарь в таблицу pandas DataFrame.
channels = pd.DataFrame(channels_data)

# Показываем справочник каналов.
display(channels)

## 6. Сохраняем таблицы в разные форматы

- обращения — CSV;
- клиенты — XLSX, если доступен `openpyxl`;
- каналы — JSON.

Если Excel недоступен, клиенты сохраняются в резервный CSV-файл.

In [ ]:
# Создаём путь к CSV-файлу с обращениями.
tickets_path = data_dir / "tickets.csv"

# Создаём путь к JSON-файлу с каналами.
channels_path = data_dir / "channels.json"

# Сохраняем обращения в CSV без дополнительного столбца индекса.
tickets.to_csv(tickets_path, index=False, encoding="utf-8-sig")

# Сохраняем каналы в JSON как список записей.
channels.to_json(channels_path, orient="records", force_ascii=False, indent=2)

# Проверяем, доступна ли работа с Excel.
if excel_available:
    # Создаём путь к Excel-файлу с клиентами.
    clients_path = data_dir / "clients.xlsx"

    # Сохраняем справочник клиентов в Excel без дополнительного столбца индекса.
    clients.to_excel(clients_path, index=False)

    # Сообщаем, какой файл создан.
    print("Создан Excel-файл:", clients_path)

# Если Excel недоступен, используем резервный CSV.
else:
    # Создаём путь к резервному CSV-файлу с клиентами.
    clients_path = data_dir / "clients.csv"

    # Сохраняем справочник клиентов в CSV.
    clients.to_csv(clients_path, index=False, encoding="utf-8-sig")

    # Сообщаем, какой резервный файл создан.
    print("Создан резервный CSV-файл:", clients_path)

# Сообщаем о создании CSV-файла обращений.
print("Создан CSV-файл:", tickets_path)

# Сообщаем о создании JSON-файла каналов.
print("Создан JSON-файл:", channels_path)

## 7. Проверяем созданные файлы

Этот блок показывает, какие файлы появились в папке `data`.

In [ ]:
# Перебираем все файлы в папке data в алфавитном порядке.
for file_path in sorted(data_dir.iterdir()):
    # Показываем имя каждого найденного файла.
    print(file_path.name)

## 8. Удаляем таблицы из памяти

Это учебный шаг. После него мы не сможем использовать старые DataFrame и будем вынуждены загрузить данные именно из файлов.

In [ ]:
# Удаляем таблицу обращений из памяти.
del tickets

# Удаляем таблицу клиентов из памяти.
del clients

# Удаляем таблицу каналов из памяти.
del channels

# Сообщаем, что исходные DataFrame удалены.
print("Исходные таблицы удалены из памяти.")

## 9. Загружаем данные из файлов

Теперь используем разные функции pandas:

- `read_csv()` для CSV;
- `read_excel()` для Excel;
- `read_json()` для JSON.

In [ ]:
# Загружаем обращения из CSV-файла.
tickets = pd.read_csv(tickets_path)

# Проверяем расширение файла со справочником клиентов.
if clients_path.suffix == ".xlsx":
    # Загружаем клиентов из Excel-файла.
    clients = pd.read_excel(clients_path)

# Если используется резервный CSV, выполняется этот блок.
else:
    # Загружаем клиентов из CSV-файла.
    clients = pd.read_csv(clients_path)

# Загружаем каналы из JSON-файла.
channels = pd.read_json(channels_path)

# Показываем загруженную таблицу обращений.
print("Обращения:")
display(tickets)

# Показываем загруженный справочник клиентов.
print("Клиенты:")
display(clients)

# Показываем загруженный справочник каналов.
print("Каналы:")
display(channels)

## 10. Выполняем предварительную проверку

Перед объединением важно убедиться, что данные действительно загрузились и имеют ожидаемую структуру.

In [ ]:
# Показываем количество строк и столбцов в таблице обращений.
print("Размер таблицы обращений:", tickets.shape)

# Показываем названия столбцов таблицы обращений.
print("Столбцы обращений:", tickets.columns.tolist())

# Показываем типы данных в таблице обращений.
print("Типы данных:")
print(tickets.dtypes)

# Считаем пропущенные значения в каждом столбце.
print("Пропущенные значения:")
print(tickets.isna().sum())

## 11. Преобразуем дату

После чтения CSV дата обычно загружается как текст. Преобразуем столбец в специальный тип даты и времени.

In [ ]:
# Преобразуем текстовые даты в тип datetime.
tickets["Дата обращения"] = pd.to_datetime(tickets["Дата обращения"])

# Показываем новый тип столбца с датой.
print("Новый тип столбца:", tickets["Дата обращения"].dtype)

# Показываем первые строки после преобразования.
display(tickets.head())

## 12. Объединяем обращения с клиентами

Общий столбец — `Код клиента`. Используем левое объединение, чтобы сохранить все обращения, даже если клиент не найден в справочнике.

In [ ]:
# Объединяем таблицу обращений со справочником клиентов.
integrated_data = tickets.merge(
    # Указываем таблицу, которую присоединяем.
    clients,

    # Указываем общий столбец для объединения.
    on="Код клиента",

    # Сохраняем все строки основной таблицы обращений.
    how="left"
)

# Показываем результат первого объединения.
display(integrated_data)

## 13. Присоединяем названия каналов

Общий столбец — `Код канала`.

In [ ]:
# Объединяем текущую таблицу со справочником каналов.
integrated_data = integrated_data.merge(
    # Указываем таблицу, которую присоединяем.
    channels,

    # Указываем общий столбец для объединения.
    on="Код канала",

    # Сохраняем все обращения.
    how="left"
)

# Показываем итоговую объединённую таблицу.
display(integrated_data)

## 14. Проверяем результат объединения

Количество обращений не должно измениться. Также найдём обращение, для которого сведения о клиенте не были найдены.

In [ ]:
# Показываем количество строк в исходной таблице обращений.
print("Строк до объединения:", len(tickets))

# Показываем количество строк после объединения.
print("Строк после объединения:", len(integrated_data))

# Выбираем строки, где сегмент клиента не заполнен.
missing_clients = integrated_data[
    integrated_data["Сегмент клиента"].isna()
]

# Показываем обращения с неизвестным клиентом.
print("Обращения, для которых клиент не найден:")
display(missing_clients)

## 15. Выполняем простой расчёт

Посчитаем количество обращений по каждому каналу связи.

In [ ]:
# Считаем, сколько раз встречается каждое название канала.
tickets_by_channel = integrated_data["Название канала"].value_counts()

# Показываем полученный результат.
print("Количество обращений по каналам:")
print(tickets_by_channel)

## 16. Сохраняем итоговую таблицу

Сохраняем объединённые данные в новый CSV-файл и затем повторно открываем его для проверки.

In [ ]:
# Создаём путь к итоговому CSV-файлу.
result_path = output_dir / "integrated_tickets.csv"

# Сохраняем объединённую таблицу без дополнительного столбца индекса.
integrated_data.to_csv(result_path, index=False, encoding="utf-8-sig")

# Загружаем сохранённый файл обратно для проверки.
result_check = pd.read_csv(result_path)

# Показываем путь к сохранённому файлу.
print("Итоговый файл сохранён:", result_path)

# Показываем размер повторно загруженной таблицы.
print("Размер сохранённой таблицы:", result_check.shape)

# Показываем повторно загруженную таблицу.
display(result_check)

## Итоги основной части

Мы выполнили полный базовый цикл:

1. создали небольшие таблицы вручную;
2. сохранили их в разные форматы;
3. загрузили CSV, Excel/CSV и JSON;
4. проверили структуру данных;
5. преобразовали дату;
6. объединили таблицы по ключам;
7. нашли несовпавший ключ;
8. выполнили простой расчёт;
9. сохранили итоговый результат.

### Контрольные вопросы

1. Почему после сохранения мы удалили DataFrame из памяти?
2. Какая функция загружает CSV?
3. Какая функция загружает Excel?
4. Какая функция загружает JSON?
5. Что означает `how="left"`?
6. Почему у клиента `КЛ-4` появились пропуски?
7. Что нужно проверить после объединения таблиц?

# Самостоятельное мини-задание

Выполните три шага:

1. посчитайте количество обращений по статусам;
2. выберите только обращения со статусом `Решено`;
3. сохраните их в файл `outputs/resolved_tickets.csv`.

Подсказки:

- для подсчёта используйте `value_counts()`;
- для отбора используйте условие в квадратных скобках;
- для сохранения используйте `to_csv()`.

In [ ]:
# Напишите решение мини-задания ниже.